## Data processing

In [ ]:
#抓資料
import pyodbc
import pandas as pd

# 連線到 SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Power114;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"  # 信任伺服器憑證
)

# 範例：讀取 11401meter ID=4
df = pd.read_sql("SELECT * FROM dbo.[11401meter] WHERE ID = 4", conn)
print(df.head())

# 要處理的表名列表
tables = [f"{i:05d}meter" for i in range(11401, 11413)]  # 11401meter ~ 11412meter

for table in tables:
    # SQL 查詢 ID = 4 的資料
    query = f"SELECT * FROM dbo.[{table}] WHERE ID = 4"
    
    # 讀取資料
    df = pd.read_sql(query, conn)
    
    # 匯出 CSV，避免中文亂碼
    csv_filename = f"{table}_ID4.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    
    print(f"{csv_filename} 已匯出，資料列數: {len(df)}")


C:\Users\rayhu\AppData\Local\Temp\ipykernel_6108\2088552883.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM dbo.[11401meter] WHERE ID = 4", conn)


        WDATE  WTIME  ID      Vrs      Vst      Vtr    Ir     Is     It    wr  \
0  2025/01/01  18:15   4  10856.4  10899.1  10967.5  8.30   9.38   8.49  None   
1  2025/01/01  18:30   4  10883.4  10927.4  10992.5  8.49   9.88   9.64  None   
2  2025/01/01  18:45   4  10880.8  10914.3  10992.3  7.93   9.29   8.45  None   
3  2025/01/01  19:00   4  10881.9  10916.9  10985.6  9.41  10.84  10.79  None   
4  2025/01/01  19:15   4  10866.8  10910.8  10970.8  7.28   8.60   8.43  None   

   ... 離KWH 週六半KWH   流動電費    CLASS    契約需量 Runtimekwh  KVAH KVARH     MINkwh  \
0  ...  NaN    NaN  107.4  平日半尖峰時段  2500.0  2772161.0  None   0.0  2772120.0   
1  ...  NaN    NaN  120.5  平日半尖峰時段  2500.0  2772207.0  None   0.0  2772161.0   
2  ...  NaN    NaN  100.3  平日半尖峰時段  2500.0  2772245.3  None   0.0  2772207.0   
3  ...  NaN    NaN  107.4  平日半尖峰時段  2500.0  2772286.3  None   0.0  2772245.0   
4  ...  NaN    NaN  119.2  平日半尖峰時段  2500.0  2772331.8  None   0.0  2772286.0   

   uploaded  
0      None  
1   

C:\Users\rayhu\AppData\Local\Temp\ipykernel_6108\2088552883.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


11401meter_ID4.csv 已匯出，資料列數: 2910
11402meter_ID4.csv 已匯出，資料列數: 2673
11403meter_ID4.csv 已匯出，資料列數: 2975
11404meter_ID4.csv 已匯出，資料列數: 2807
11405meter_ID4.csv 已匯出，資料列數: 2944
11406meter_ID4.csv 已匯出，資料列數: 2877
11407meter_ID4.csv 已匯出，資料列數: 2976
11408meter_ID4.csv 已匯出，資料列數: 2953
11409meter_ID4.csv 已匯出，資料列數: 2879
11410meter_ID4.csv 已匯出，資料列數: 2849
11411meter_ID4.csv 已匯出，資料列數: 2747
11412meter_ID4.csv 已匯出，資料列數: 2955


In [ ]:
import pandas as pd
import os

# CSV 檔案路徑列表
csv_files = [f"{i:05d}meter_ID4.csv" for i in range(11401, 11413)]

# 建立一個空的 DataFrame 來合併
combined_df = pd.DataFrame()

for file in csv_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        # 可選：加一欄標記來源表格
        df['SourceTable'] = file.replace('_ID4.csv','')
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    else:
        print(f"{file} 不存在，跳過")

# 匯出合併後的 CSV，保持中文不亂碼
combined_df.to_csv("114_ID4.csv", index=False, encoding='utf-8-sig')

print("已成功匯出114_ID4.csv")


已成功匯出114_ID4.csv


In [26]:
import pandas as pd

# 讀取原始檔，指定編碼避免亂碼
df = pd.read_csv("114_ID4.csv", encoding="utf-8-sig")

# 轉成 datetime（避免字串排序錯誤）
df["WDATE"] = pd.to_datetime(df["WDATE"], errors="coerce")

# 排序
df_sorted = df.sort_values(by=["WDATE", "WTIME"])

# 重新編排 index
df_sorted = df_sorted.reset_index(drop=True)

# 輸出成新檔，指定編碼避免 Excel 開啟跑版
df_sorted.to_csv("114_ID4_sorted.csv", index=False, encoding="utf-8-sig")

print("排序完成 ✅ 已輸出 114_ID4_sorted.csv")


排序完成 ✅ 已輸出 114_ID4_sorted.csv


In [2]:
import pandas as pd

# 讀取 CSV
df = pd.read_csv("114_ID4_kwh.csv", encoding="utf-8-sig")

# 轉成日期格式
df["WDATE"] = pd.to_datetime(df["WDATE"], errors="coerce")

# 只取日期
df["date"] = df["WDATE"].dt.date

# 按日期累加 kwh
daily_kwh = df.groupby("date")["kwh"].sum().round(2).reset_index()

# 再轉回 datetime
daily_kwh["date"] = pd.to_datetime(daily_kwh["date"])

# 星期簡寫對照
weekday_map = {
    0: "一",
    1: "二",
    2: "三",
    3: "四",
    4: "五",
    5: "六",
    6: "日",
}

daily_kwh["星期"] = daily_kwh["date"].dt.weekday.map(weekday_map)

# 輸出
daily_kwh.to_csv("114_dailykwh.csv", index=False, encoding="utf-8-sig")

print("完成 ✅ 已生成 114_dailykwh.csv（含星期簡寫欄位）")


完成 ✅ 已生成 114_dailykwh.csv（含星期簡寫欄位）


In [27]:
import pandas as pd

# 1️⃣ 讀取日資料
df = pd.read_csv("114_dailykwh_cleaned.csv")

# 2️⃣ 轉成日期格式
df["date"] = pd.to_datetime(df["date"])

# 3️⃣ 建立 year / week（ISO 週）
df["year"] = df["date"].dt.isocalendar().year
df["week"] = df["date"].dt.isocalendar().week

# 4️⃣ 週聚合（加入高低溫）
weekly_df = df.groupby(["year", "week"]).agg({
    "kwh": "sum",              # 週總用電
    "avg_temp": "mean",        # 週平均溫度
    "high_temp": "max",        # 週最高溫
    "low_temp": "min",         # 週最低溫
    "is_holiday": "sum",       # 本週假日天數
    "is_vacation": "sum"       # 本週寒暑假天數
}).reset_index()

# 5️⃣ 重新命名欄位
weekly_df = weekly_df.rename(columns={
    "kwh": "week_kwh",
    "avg_temp": "week_avg_temp",
    "high_temp": "week_high_temp",
    "low_temp": "week_low_temp",
    "is_holiday": "holiday_days",
    "is_vacation": "vacation_days"
})

# 6️⃣ 依照時間排序
weekly_df = weekly_df.sort_values(["year", "week"])

# 7️⃣ 存成新的 CSV
weekly_df.to_csv("114_weeklykwh.csv", index=False, encoding="utf-8-sig")

print("✅ 週資料已成功產生")
print(weekly_df.head())


✅ 週資料已成功產生
   year  week  week_kwh  week_avg_temp  week_high_temp  week_low_temp  \
0  2025     1   19805.3      17.400000              21             16   
1  2025     2   25957.5      14.714286              19             12   
2  2025     3   22100.5      15.928571              24             12   
3  2025     4   21900.6      18.214286              23             12   
4  2025     5   11867.2      16.214286              23              9   

   holiday_days  vacation_days  
0             1              0  
1             0              0  
2             0              7  
3             0              7  
4             6              7  


In [26]:
import pandas as pd

# 1️⃣ 讀取日資料
df = pd.read_csv("114_dailykwh_cleaned.csv")

# 2️⃣ 轉成日期格式
df["date"] = pd.to_datetime(df["date"])

# 3️⃣ 取年月，作為月份
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# 4️⃣ 月聚合
monthly_df = df.groupby(["year", "month"]).agg({
    "kwh": "sum",               # 月總電量
    "avg_temp": "mean",         # 月平均溫度
    "is_holiday": "sum",        # 月假日天數
    "is_vacation": "sum"        # 月寒暑假天數
}).reset_index()

# 5️⃣ 重新命名欄位
monthly_df = monthly_df.rename(columns={
    "kwh": "month_kwh",
    "avg_temp": "month_avg_temp",
    "is_holiday": "holiday_days",
    "is_vacation": "vacation_days"
})

# 6️⃣ 按年月排序
monthly_df = monthly_df.sort_values(["year", "month"])

# 7️⃣ 存成 CSV
monthly_df.to_csv("114_monthlykwh.csv", index=False, encoding="utf-8-sig")

print("✅ 月資料已生成完成")
print(monthly_df.head())


✅ 月資料已生成完成
   year  month  month_kwh  month_avg_temp  holiday_days  vacation_days
0  2025      1    98027.9       16.306452             5             19
1  2025      2     6135.0       16.428571             5             16
2  2025      3     8529.1       18.887097             1              0
3  2025      4   148987.7       22.783333             5              0
4  2025      5   198640.2       26.064516             2              0


In [ ]:
import pandas as pd

# 讀取資料
df = pd.read_csv("114_dailykwh.csv")

# 原始欄位名稱
df.columns = ["date", "kwh", "weekday_ch", "holiday_type"]

# 日期轉換
df["date"] = pd.to_datetime(df["date"])

# ===== 拆解日期 =====
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# ===== 週數（星期一為一週第一天）=====
df["week"] = df["date"].dt.isocalendar().week

# ===== 星期（星期一=1, 星期日=7）=====
df["weekday"] = df["date"].dt.weekday + 1

# ===== 放假邏輯 =====
# 只有 holiday_type 有值才算放假
df["is_holiday"] = 0
df.loc[df["holiday_type"].notna(), "is_holiday"] = 1

# ===== 寒暑假單獨拉出 =====
df["is_vacation"] = 0
df.loc[df["holiday_type"].isin(["寒假", "暑假"]), "is_vacation"] = 1

# ===== 溫度欄位（先留空）=====
df["high_temp"] = None
df["low_temp"] = None
df["avg_temp"] = None

# ===== 欄位排序 =====
df = df[
    [
        "date",
        "year",
        "month",
        "day",
        "week",
        "weekday",
        "is_holiday",
        "is_vacation",
        "high_temp",
        "low_temp",
        "avg_temp",
        "kwh",
    ]
]

# 輸出
df.to_csv("114_dailykwh_cleaned.csv", index=False)

print("整理完成")


FileNotFoundError: [Errno 2] No such file or directory: '114_dailykwh.csv'

In [2]:
import pandas as pd

# 讀取資料
df = pd.read_csv("114_dailykwh_cleaned.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# ===== 週數
df["kwh_lag1"] = df["kwh"].shift(1)
df["kwh_lag7"] = df["kwh"].shift(7)
df["rolling7"] = df["kwh"].shift(1).rolling(window=7).mean()
df = df.dropna(subset=["kwh_lag1", "kwh_lag7", "rolling7"]).reset_index(drop=True)
df["avg_temp_sq"] = df["avg_temp"] ** 2

# ===== 欄位排序 =====
df = df[
    [
        "date",
        "year",
        "month",
        "day",
        "week",
        "weekday",
        "is_holiday",
        "is_vacation",
        "high_temp",
        "low_temp",
        "avg_temp",
        "avg_temp_sq",
        "kwh_lag1",
        "kwh_lag7",
        "rolling7",
        "kwh",
    ]
]

# 輸出
df.to_csv("114_dailykwh_cleaned1.csv", index=False)

print("整理完成")
print(df.head(10))

整理完成
        date  year  month  day  week  weekday  is_holiday  is_vacation  \
0 2025-01-08  2025      1    8     2        3           0            0   
1 2025-01-09  2025      1    9     2        4           0            0   
2 2025-01-10  2025      1   10     2        5           0            0   
3 2025-01-11  2025      1   11     2        6           0            0   
4 2025-01-12  2025      1   12     2        7           0            0   
5 2025-01-13  2025      1   13     3        1           0            1   
6 2025-01-14  2025      1   14     3        2           0            1   
7 2025-01-15  2025      1   15     3        3           0            1   
8 2025-01-16  2025      1   16     3        4           0            1   
9 2025-01-17  2025      1   17     3        5           0            1   

   high_temp  low_temp  avg_temp  avg_temp_sq  kwh_lag1  kwh_lag7  \
0         17        15      16.0       256.00    4017.2    3609.2   
1         16        12      14.0       196

## Day Linear Regression


In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1️⃣ 讀取資料
df = pd.read_csv("114_dailykwh_cleaned.csv")

# 2️⃣ 排序（時間序列一定要排序）
df = df.sort_values("date").reset_index(drop=True)

# 3️⃣ 如果有缺值先刪除（避免報錯）
df = df.dropna()

# 4️⃣ 選擇特徵（加入 month）
features = ["weekday", "month", "is_holiday", "is_vacation", "avg_temp"]
target = "kwh"

X = df[features]
y = df[target]

# 5️⃣ 前 70% 訓練，後 30% 測試
split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# 6️⃣ 建立模型
model = LinearRegression()
model.fit(X_train, y_train)

# 7️⃣ 預測
y_pred = model.predict(X_test)

# 8️⃣ 評估指標
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== 模型評估結果 ===")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")

# 9️⃣ 查看各特徵係數
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

print("\n=== 特徵係數 ===")
print(coef_df)

print(f"\nIntercept (截距): {model.intercept_:.2f}")


=== 模型評估結果 ===
MAE  : 2012.37
RMSE : 2187.62
R²   : -0.202

=== 特徵係數 ===
       Feature  Coefficient
0      weekday  -304.893729
1        month   210.056685
2   is_holiday  -985.121314
3  is_vacation   113.538375
4     avg_temp   255.996595

Intercept (截距): -1482.71


In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1️⃣ 讀取資料
df = pd.read_csv("114_dailykwh_cleaned.csv")

# 2️⃣ 移除缺值
df = df.dropna()

# 3️⃣ 特徵與目標
features = ["weekday", "month", "is_holiday", "is_vacation", "avg_temp"]
target = "kwh"

X = df[features]
y = df[target]

# 4️⃣ 隨機 50% 訓練 / 50% 測試
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.5,
    random_state=42
)

# 5️⃣ 建立模型
model = LinearRegression()
model.fit(X_train, y_train)

# 6️⃣ 預測
y_pred = model.predict(X_test)

# 7️⃣ 評估
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== 隨機切分模型結果 ===")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")


=== 隨機切分模型結果 ===
MAE  : 1581.75
RMSE : 1905.39
R²   : 0.429


## Week Linear Regression

In [28]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1️⃣ 讀取週資料
df = pd.read_csv("114_weeklykwh.csv")

# 2️⃣ 特徵與目標
X = df[["week_avg_temp", "holiday_days", "vacation_days"]]
y = df["week_kwh"]

# 3️⃣ 時間序列切分：前 70% 訓練 / 後 30% 測試
split_index = int(len(df) * 0.7)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# 4️⃣ 建立模型
model = LinearRegression()
model.fit(X_train, y_train)

# 5️⃣ 預測
y_pred = model.predict(X_test)

# 6️⃣ 評估
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== 週資料（無 week 特徵） ===")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")


=== 週資料（無 week 特徵） ===
MAE  : 9641.40
RMSE : 10897.74
R²   : -0.494


In [31]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1️⃣ 讀取週資料
df = pd.read_csv("114_weeklykwh.csv")

# 2️⃣ 特徵與目標
X = df[["week_avg_temp", "holiday_days", "vacation_days"]]
y = df["week_kwh"]

# 3️⃣ 隨機 50% 訓練 / 50% 測試
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.5,
    random_state=42
)

# 4️⃣ 建立模型
model = LinearRegression()
model.fit(X_train, y_train)

# 5️⃣ 預測
y_pred = model.predict(X_test)

# 6️⃣ 評估
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== 週資料（隨機 50% 切分） ===")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")


=== 週資料（隨機 50% 切分） ===
MAE  : 9220.10
RMSE : 11011.87
R²   : 0.546


In [32]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1️⃣ 讀取週資料
df = pd.read_csv("114_weeklykwh.csv")

# 2️⃣ 特徵與目標
X = df[[
    "week_avg_temp",    # 平均溫度
    "week_high_temp",   # 最高溫
    "week_low_temp",    # 最低溫
    "holiday_days",     # 假日天數
    "vacation_days"     # 寒暑假天數
]]
y = df["week_kwh"]

# 3️⃣ 隨機 50% 訓練 / 50% 測試
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.5,
    random_state=42
)

# 4️⃣ 建立模型
model = LinearRegression()
model.fit(X_train, y_train)

# 5️⃣ 預測
y_pred = model.predict(X_test)

# 6️⃣ 評估
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== 週資料（隨機 50% 切分） ===")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")


=== 週資料（隨機 50% 切分） ===
MAE  : 9178.72
RMSE : 11438.32
R²   : 0.510


## Month Linear Regression

## clean1.csv

In [11]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =========================
# 1️⃣ 讀取資料
# =========================
df = pd.read_csv("114_dailykwh_cleaned1.csv")

print("資料筆數：", len(df))


# =========================
# 4️⃣ 設定 X 與 y
# =========================
features = [
    "month",
    "weekday",
    "is_holiday",
    "is_vacation",
    "avg_temp",
    "avg_temp_sq",
    "kwh_lag1",
    "kwh_lag7",
    "rolling7"
]

X = df[features]
y = df["kwh"]


# =========================
# 5️⃣ 時間切分（70% / 30%）
# =========================
split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]


# =========================
# 6️⃣ 建立線性回歸模型
# =========================
model = LinearRegression()
model.fit(X_train, y_train)


# =========================
# 7️⃣ 預測
# =========================
y_pred = model.predict(X_test)


# =========================
# 8️⃣ 評估模型
# =========================
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2:", round(r2, 4))
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))


# =========================
# 9️⃣ 看各特徵影響程度
# =========================
# print("\n特徵係數：")
# for name, coef in zip(features, model.coef_):
#     print(f"{name:15s} {coef:.4f}")


# =========================
# 🔟 畫圖（真實 vs 預測）
# =========================
# plt.figure(figsize=(12,5))
# plt.plot(y_test.values, label="Actual")
# plt.plot(y_pred, label="Predicted")
# plt.legend()
# plt.title("Actual vs Predicted")
# plt.show()

資料筆數： 358
R2: 0.5962
MAE: 881.54
RMSE: 1182.96


In [7]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# =========================
# 1️⃣ 讀取資料
# =========================
df = pd.read_csv("114_dailykwh_cleaned1.csv")

print("資料筆數：", len(df))


# =========================
# ⭐ 新增週期特徵（最重要）
# =========================

# 星期週期特徵
df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

# 月份週期特徵
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


# =========================
# 4️⃣ 設定 X 與 y
# =========================

features = [
    "is_holiday",
    "is_vacation",
    "avg_temp",
    "avg_temp_sq",
    "kwh_lag1",
    "kwh_lag7",
    "rolling7",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos"
]

X = df[features]
y = df["kwh"]


# =========================
# 5️⃣ 時間切分（70% / 30%）
# =========================

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]


# =========================
# 6️⃣ 建立線性回歸模型
# =========================

model = LinearRegression()
model.fit(X_train, y_train)


# =========================
# 7️⃣ 預測
# =========================

y_pred = model.predict(X_test)


# =========================
# 8️⃣ 評估模型
# =========================

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2:", round(r2, 4))
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

資料筆數： 358
R2: 0.5286
MAE: 937.1
RMSE: 1278.04
